This notebook allows to visualise one datasets

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="darkgrid")
sns.set(font_scale = 1.5)

from generate import *

## Data generation

In [ ]:
x, t, e, betas, outcomes = generate()

In [ ]:
e.value_counts()

In [ ]:
sns.scatterplot(x=x[0], y=x[1], hue=outcomes.cluster+1, alpha = 0.33, palette = ['olive', 'orange'])
plt.title("Cluster")
plt.xlabel(r'$x_1$')
plt.ylabel(r'$x_2$')
plt.legend(title = 'Cluster', loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
def cumdensity(data, label = 'CDF'):
    count, bins_count = np.histogram(data, bins = 100)
    pdf = count / sum(count)
    cdf = np.cumsum(pdf)
    plt.plot(bins_count[1:], cdf, label = label)    

In [ ]:
for event in range(3):
    for cluster in range(2):
        cumdensity(outcomes['duration'][(outcomes.cluster == cluster) & (outcomes['event'] == event)], label = 'Cluster {}'.format(cluster))
    plt.title('Event {}'.format(event))
    plt.ylabel(r'Obsevred distribution $T | D = $ {}'.format(event))
    plt.xlabel('Time')
    plt.xlim(0, 2)
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

In [ ]:
cif = compute_cif(x, betas, outcomes.cluster, np.linspace(0, outcomes.duration.max(), 1000))

In [ ]:
cif[1].iloc[:50].T.plot(ls = '-.', legend = False)
cif[1].mean().plot(lw = 5)

In [ ]:
for event in range(2):
    for cluster in range(2):
        cif[event + 1][outcomes.cluster == cluster].mean(0).plot(label = 'Cluster {}'.format(cluster))
    plt.title('Event {}'.format(event + 1))
    plt.ylabel(r'Distribution $T_r$\'')
    plt.xlabel('Time')
    plt.xlim(0, 2)
    plt.ylim(0, 1)
    plt.legend()
    plt.show()

In [ ]:
outcomes.event.groupby(outcomes.cluster).value_counts()

In [ ]:
plt.hist(t, bins = 100)

In [ ]:
differences, censoring = [], []
for i in range(1, 51):
    x, _, _, betas, outcomes = generate(i)
    total = outcomes.event.groupby(outcomes.cluster).value_counts()
    censoring.append((outcomes.event == 0).mean())

    shape_x = {event: shape[event](betas[event][x.z.values], x.drop(columns = 'z').values) for event in [1, 2]}
    hazard_ratio = shape_x[1] / (shape_x[1] + shape_x[2])
    differences.append(hazard_ratio[outcomes.cluster == 0].mean() - hazard_ratio[outcomes.cluster == 1].mean())
plt.hist(differences)
plt.xlabel('Theoretical bias')
plt.ylabel('Count')
plt.show()

plt.hist(censoring)
plt.xlabel('Censoring rate')
plt.ylabel('Count')